# Quick architecture follow-up

Four full-model runs: 7/21-day history × one/two shared convolution stages. One seed, up to five epochs, patience two. Training contexts start on day 21 for every variant, keeping dates and targets identical. Both weekly-profile baselines appear in the results.


In [ ]:
import importlib.util
import json
import os
import sys
from pathlib import Path

# Optional explicit override, e.g. Path('/workspace/trans_predicting').
PROJECT_ROOT = os.environ.get('TRAM_FORECAST_ROOT')

def find_project_root(explicit=None):
    if explicit:
        candidates = [Path(explicit).expanduser()]
    else:
        cwd = Path.cwd()
        candidates = [cwd, *cwd.parents, Path('/workspace/trans_predicting')]
        spec = importlib.util.find_spec('tram_forecast')
        if spec is not None and spec.origin:
            candidates.extend(Path(spec.origin).resolve().parents)
    for candidate in candidates:
        candidate = candidate.resolve()
        if (candidate / 'src/tram_forecast').is_dir() and (
            candidate / 'configs/experiments/quick_followup.json'
        ).is_file():
            return candidate
    checked = ', '.join(str(p) for p in candidates)
    raise FileNotFoundError(
        'Cannot find the tram_forecast project with configs/experiments/quick_followup.json. '
        'Sync src/ and configs/ to /workspace/trans_predicting, or set PROJECT_ROOT '
        f'to your remote project directory. Checked: {checked}'
    )

ROOT = find_project_root(PROJECT_ROOT)
sys.path.insert(0, str(ROOT / 'src'))
from tram_forecast.experiments import run_suite
from tram_forecast.io import write_json

print(f'Project directory: {ROOT}')
suite = json.loads((ROOT / 'configs/experiments/quick_followup.json').read_text())
ARTIFACT = ROOT / 'outputs/prepared/validation'
OUTPUT = ROOT / 'outputs/architecture/quick_followup_v1'
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cuda':
    gpu = torch.cuda.get_device_properties(0)
    print(f'Device: {gpu.name} | VRAM: {gpu.total_memory / 2**30:.1f} GiB')
else:
    print('CUDA unavailable: training will use CPU. Check the remote kernel PyTorch installation.')


In [ ]:
# Edit these before previewing/training. All variants share the training settings.
suite['base']['training']['forecast_days'] = 7
suite['base']['training']['epochs'] = 5
suite['base']['training']['patience'] = 2
suite['base']['training']['batch_size'] = 64
suite['base']['training']['accumulation'] = 1
# Larger event chunks reduce the number of small GPU operations.
suite['base']['model']['max_hours'] = 256
suite['base']['model']['max_positions'] = 262144
suite['base']['model']['checkpoint_events'] = False
suite['base']['training']['device'] = DEVICE
# If VRAM remains low, try batch_size=128. For OOM, reduce batch/chunk sizes.
suite['seeds'] = [67]  # One seed for a quick screen.
# Optional: keep just a few variants for a first pass.
# suite['variants'] = suite['variants'][:3]
suite['variants']


In [ ]:
SUITE_PATH = OUTPUT / 'suite.json'
write_json(SUITE_PATH, suite)
preview = run_suite(SUITE_PATH, artifact=ARTIFACT if ARTIFACT.exists() else None)


## Train

The next cell trains every selected variant sequentially. Rerunning it starts fresh: previous selected runs and their comparison are moved into OUTPUT/archive/. Set overwrite=False to refuse existing runs instead. Completed results are saved after every variant. To continue an interrupted run rather than restart, use the normal training CLI with its saved config.json and --resume latest.pt.


In [ ]:
# Rerunning this cell archives previous selected runs and trains from scratch.
results = run_suite(
    SUITE_PATH, artifact=ARTIFACT, output=OUTPUT, device=DEVICE,
    run=True, overwrite=True,
)


In [ ]:
from IPython.display import Markdown, display

results = json.loads((OUTPUT / 'comparison.json').read_text())
baselines = json.loads((OUTPUT / 'baselines.json').read_text())
lines = ['| Model | History days | Parameters | WAPE ↓ | MAE ↓ | Best epoch | Train seconds |',
         '|---|---:|---:|---:|---:|---:|---:|']
for b in baselines['baselines']:
    lines.append(f"| Weekly profile | {b['history_days']} | 0 | {b['wape']:.2%} | {b['mae']:.4f} | — | — |")
for r in results:
    lines.append(f"| {r['name']} | {r['history_days']} | {r['parameters']:,} | {r['wape']:.2%} | {r['mae']:.4f} | {r['best_epoch']} | {r['training_seconds']:.1f} |")
display(Markdown('\n'.join(lines)))
